## EDA Paso 1: Exploración Inicial

In [0]:
-- ============================================================
-- EDA 1.1: ¿Cuántos registros tenemos?
-- ============================================================

SELECT COUNT(*) as total_registros
FROM analisis_tmdb.bronze.movies_bronze

In [0]:
-- ============================================================
-- EDA 1.2: Ver las primeras filas
-- ============================================================

SELECT *
FROM analisis_tmdb.bronze.movies_bronze
LIMIT 10;

## EDA Paso 2: Análisis de Valores Nulos

In [0]:
-- ============================================================
-- EDA 2.1: Contar valores nulos por columna
-- ============================================================

WITH total AS (
    SELECT COUNT(*) as n 
    FROM analisis_tmdb.bronze.movies_bronze
),
nulos AS (
    SELECT
        COUNT(*) - COUNT(adult) as nulos_adult,
        COUNT(*) - COUNT(backdrop_path) as nulos_backdrop_path,
        COUNT(*) - COUNT(genre_ids) as nulos_genre_ids,
        COUNT(*) - COUNT(id) as nulos_id,
        COUNT(*) - COUNT(original_language) as nulos_original_language,
        COUNT(*) - COUNT(original_title) as nulos_original_title,
        COUNT(*) - COUNT(overview) as nulos_overview,
        COUNT(*) - COUNT(popularity) as nulos_popularity,
        COUNT(*) - COUNT(poster_path) as nulos_poster_path,
        COUNT(*) - COUNT(release_date) as nulos_release_date,
        COUNT(*) - COUNT(title) as nulos_title,
        COUNT(*) - COUNT(video) as nulos_video,
        COUNT(*) - COUNT(vote_average) as nulos_vote_average,
        COUNT(*) - COUNT(vote_count) as nulos_vote_count,
        COUNT(*) - COUNT(ingestion_timestamp) as nulos_ingestion_timestamp
    FROM analisis_tmdb.bronze.movies_bronze
)
SELECT 
    t.n as total_registros,
    n.*
FROM total t, nulos n;

In [0]:
-- ============================================================
-- EDA 2.2: Porcentaje de nulos en columnas clave
-- ============================================================

WITH totales AS (
    SELECT COUNT(*) as total FROM analisis_tmdb.bronze.movies_bronze
)
SELECT 
    ROUND((t.total - COUNT(backdrop_path)) * 100.0 / t.total, 2) as pct_nulos_backdrop_path,
    ROUND((t.total - COUNT(poster_path)) * 100.0 / t.total, 2) as pct_nulos_poster_path
FROM analisis_tmdb.bronze.movies_bronze
CROSS JOIN totales t
GROUP BY t.total;

## EDA Paso 3: Cardinalidad y Distribución de Categóricos

In [0]:
-- ============================================================
-- EDA 3.1: Distribución de original_language
-- ============================================================

SELECT 
    original_language,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM analisis_tmdb.bronze.movies_bronze
GROUP BY original_language
ORDER BY cantidad DESC;

In [0]:
-- ============================================================
-- EDA 3.2: Distribución de popularity
-- ============================================================

SELECT 
    popularity,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM analisis_tmdb.bronze.movies_bronze
GROUP BY popularity
ORDER BY cantidad DESC;

In [0]:
-- ============================================================
-- EDA 3.3: Distribución de vote_average
-- ============================================================

SELECT 
    vote_average,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM analisis_tmdb.bronze.movies_bronze
GROUP BY vote_average
ORDER BY cantidad DESC;

## EDA Paso 4: Estadísticas Descriptivas de Variables Numéricas

In [0]:
-- ============================================================
-- EDA 4.1: Estadísticas de popularity por pelicula
-- ============================================================

SELECT 
    title,
    COUNT(*) as cantidad,
    ROUND(MIN(popularity), 2) as popularity_min,
    ROUND(MAX(popularity), 2) as popularity_max,
    ROUND(AVG(popularity), 2) as popularity_promedio,
    ROUND(PERCENTILE(popularity, 0.5), 2) as popularity_mediana,
    ROUND(PERCENTILE(popularity, 0.25), 2) as popularity_p25,
    ROUND(PERCENTILE(popularity, 0.75), 2) as popularity_p75
FROM analisis_tmdb.bronze.movies_bronze
WHERE popularity IS NOT NULL AND popularity > 0
group by title
ORDER BY cantidad DESC;

In [0]:
-- ============================================================
-- EDA 4.2: Estadísticas de vote_average por pelicula
-- ============================================================

SELECT 
    title,
    COUNT(*) as cantidad,
    ROUND(MIN(vote_average), 2) as vote_average_min,
    ROUND(MAX(vote_average), 2) as vote_average_max,
    ROUND(AVG(vote_average), 2) as vote_average_promedio,
    ROUND(PERCENTILE(vote_average, 0.5), 2) as vote_average_mediana
FROM analisis_tmdb.bronze.movies_bronze
WHERE vote_average IS NOT NULL AND vote_average > 0
group by title
ORDER BY cantidad DESC;

In [0]:
-- ============================================================
-- EDA 4.1: Estadísticas de vote_count por pelicula
-- ============================================================

SELECT 
    title,
    COUNT(*) as cantidad,
    ROUND(MIN(vote_count), 2) as vote_count_min,
    ROUND(MAX(vote_count), 2) as vote_count_max,
    ROUND(AVG(vote_count), 2) as vote_count_promedio,
    ROUND(PERCENTILE(vote_count, 0.5), 2) as vote_count_mediana
FROM analisis_tmdb.bronze.movies_bronze
WHERE vote_count IS NOT NULL AND vote_count > 0
group by title
ORDER BY cantidad DESC;

## EDA Paso 5: Detección de Problemas de Calidad

In [0]:
-- ============================================================
-- EDA 5.1: Reporte de calidad de datos usando CTEs
-- ============================================================

WITH total AS (
    SELECT COUNT(*) as n 
    FROM analisis_tmdb.bronze.movies_bronze
),
problemas AS (
    SELECT
        -- Popularity problemáticos 
        COUNT(CASE WHEN popularity IS NULL OR popularity <= 0 THEN 1 END) as popularity_invalido,
        -- ID vacío
        COUNT(CASE WHEN id IS NULL THEN 1 END) as id_vacio,
        -- Titulo vacío
        COUNT(CASE WHEN title IS NULL OR title = '' THEN 1 END) as titulo_vacio,
        -- Release date vacío
        COUNT(CASE WHEN release_date IS NULL OR release_date = '' THEN 1 END) as date_vacio,
        -- Genre date vacío
        COUNT(CASE WHEN cardinality(genre_ids) = 0 OR genre_ids IS NULL THEN 1 END) as genre_vacio
    FROM analisis_tmdb.bronze.movies_bronze
)
SELECT 
    t.n as total_registros,
    p.popularity_invalido,
    ROUND(p.popularity_invalido * 100.0 / t.n, 2) as pct_popularity_invalido,
    p.id_vacio,
    ROUND(p.id_vacio * 100.0 / t.n, 2) as pct_id_vacio,
    p.titulo_vacio,
    ROUND(p.titulo_vacio * 100.0 / t.n, 2) as pct_titulo_vacio,
    p.date_vacio,
    ROUND(p.date_vacio * 100.0 / t.n, 2) as pct_date_vacio,
    p.genre_vacio,
    ROUND(p.genre_vacio * 100.0 / t.n, 2) as pct_genre_vacio
FROM total t, problemas p;

In [0]:
-- ============================================================
-- EDA 5.2: Detectar posibles duplicados
-- ============================================================

WITH duplicados AS (
    SELECT 
        id,
        COUNT(*) as veces
    FROM analisis_tmdb.bronze.movies_bronze
    GROUP BY id
    HAVING COUNT(*) > 1
)
SELECT 
    COUNT(*) as grupos_duplicados,
    SUM(veces) as total_registros_duplicados,
    SUM(veces - 1) as registros_extra_por_duplicacion
FROM duplicados;

In [0]:
-- ============================================================
-- EDA 5.3: Ver ejemplos de duplicados
-- ============================================================

WITH duplicados AS (
    SELECT 
        id,
        COUNT(*) as veces
    FROM analisis_tmdb.bronze.movies_bronze
    GROUP BY id
)
SELECT *
FROM duplicados
ORDER BY veces DESC
LIMIT 10;

In [0]:
CREATE OR REPLACE TEMP VIEW bronze_EDA AS (
SELECT
    id,
    -- Validar titulo de pelicula
    CASE
        WHEN title RLIKE '^(?!["'']).*(?<!["''])$'
        THEN title
        ELSE NULL
    END AS title,
    -- Validar formato de fecha YYYY-MM-DD
    CASE
        WHEN release_date RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$'
        THEN TO_DATE(release_date)
        ELSE NULL
    END AS release_date,
    -- Validar código de idioma (2 letras)
    CASE
        WHEN original_language RLIKE '^[a-z]{2}$'
        THEN original_language
        ELSE NULL
    END AS original_language,
    -- Validar path de poster
    CASE
        WHEN poster_path RLIKE '^/.*\\.jpg$'
        THEN poster_path
        ELSE NULL
    END AS poster_path,
    popularity,
    vote_average,
    vote_count,
    ingestion_timestamp

FROM analisis_tmdb.bronze.movies_bronze);

In [0]:
select * from bronze_EDA limit 100

# ===============================
# CONCLUSIONES
# ===============================

## Hallazgos del EDA

### Problemas de calidad identificados:

1. **Campos nulos:** Varios campos como `release_year`y `genre_ids` tienen valores nulos pero el porcentaje es minimo
2. **Posibles duplicados:** Peliculas repetidas con mismo titulo y votos
3. **Texto sucio:** La columna `title` tiene caracteres como " en el nombre de la pelicula al inicio y final

### Transformaciones necesarias para capa Silver:

- [ ] Eliminar duplicados
- [ ] Limpiar caracteres especiales en texto